In [ ]:
# ==============================================================================
# STEP 1: INITIAL SETUP & VERSIONED CONFIGURATION 🏛️⚙️
# ==============================================================================

# 1. INSTALLATIONS (only if necessary)
#!pip install -q diffusers transformers accelerate torchaudio tqdm

# 2. GOOGLE DRIVE MOUNT
from google.colab import drive
drive.mount('/content/drive')

# 3. IMPORTS & SYSTEM SETUP
import os
import time
import torch
import warnings
import glob
from accelerate import Accelerator

# --- 4. PATH CONFIGURATION (FIXED TO EXISTING RUN) ---
BASE_LOG_DIR = '/content/drive/MyDrive/MasterProject/outputs/results/training_logs'

# IMPORTANT: Here we use the folder name verified earlier!
UNIQUE_RUN_NAME = "MIMII_Gen_WideUNet_Drop10_20260126_2241"

# This is our working directory for the resume
CHECKPOINT_DIR = os.path.join(BASE_LOG_DIR, 'checkpoints', UNIQUE_RUN_NAME)
RESUME_STATE_DIR = os.path.join(CHECKPOINT_DIR, "full_train_state")

# We are not creating a new folder, we use the existing one
if os.path.exists(CHECKPOINT_DIR):
    print(f"✅ Workspace verified: {CHECKPOINT_DIR}")
else:
    print(f"❌ ERROR: Path does not exist! Please check mount.")

# NEW_METHOD_DIR remains for backward compatibility of your remaining code
NEW_METHOD_DIR = CHECKPOINT_DIR

# 5. LOAD GLOBAL STATS
STATS_PATH = '/content/drive/MyDrive/MasterProject/data/features/bearing_global_stats.pt'

if os.path.exists(STATS_PATH):
    stats = torch.load(STATS_PATH, map_location='cpu')
    # Reshape for 16-channel normalization [16, 1, 1]
    GLOBAL_MEAN = stats['mean'].view(16, 1, 1)
    GLOBAL_STD = stats['std'].view(16, 1, 1)
    print(f"✅ Global stats loaded for 'Bearing'.")
else:
    print(f"❌ ERROR: Stats file not found at: {STATS_PATH}")

# ==============================================================================
# 🔍 SANITY CHECK: EXPERIMENT IDENTITY
# ==============================================================================
print(f"\n--- 🕵️ Experiment Profile ---")
print(f"🚀 ID: {UNIQUE_RUN_NAME}")
print(f"📁 Target: {NEW_METHOD_DIR}")
print(f"💻 Hardware: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"🛠️ Mode: Hitachi replication with 10% unconditional dropout")

Mounted at /content/drive
✅ Arbeitsverzeichnis bestätigt: /content/drive/MyDrive/MasterProject/outputs/results/training_logs/checkpoints/MIMII_Gen_WideUNet_Drop10_20260126_2241
✅ Globale Statistiken für 'Bearing' geladen.

--- 🕵️ Experiment-Steckbrief ---
🚀 ID: MIMII_Gen_WideUNet_Drop10_20260126_2241
📁 Ziel: /content/drive/MyDrive/MasterProject/outputs/results/training_logs/checkpoints/MIMII_Gen_WideUNet_Drop10_20260126_2241
💻 Hardware: NVIDIA A100-SXM4-80GB
🛠️ Modus: Hitachi-Replikation mit 10% Unconditional Dropout


In [ ]:
# ==============================================================================
# STEP 2: CONFIGURATION & OOM-PROTECTION 🏛️🛡️
# ==============================================================================

# 1. PATH DEFINITIONS (Synchronized with Step 1)
BASE_DIR = '/content/drive/MyDrive/MasterProject'
LATENT_ROOT = os.path.join(BASE_DIR, 'data', 'features', 'encodec')
EMBEDDING_ROOT = os.path.join(BASE_DIR, 'data', 'text_embeddings')

# Results are saved in the specific subfolder for the Hitachi replication
CHECKPOINT_DIR = NEW_METHOD_DIR
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. TRAINING HYPERPARAMETERS (Memory-optimized for Wide-UNet)
MACHINE_TYPE = 'bearing'

# --- ⚠️ THE OOM-FIX ---
BATCH_SIZE = 16               # Reduced from 64 to 16 to save VRAM
GRADIENT_ACCUMULATION_STEPS = 4 # 16 * 4 = 64 (Effective Batch Size remains the same!)
# ----------------------

LEARNING_RATE = 1e-4
NUM_EPOCHS = 700
VALIDATION_INTERVAL = 10
MAX_GRAD_NORM = 1.0
MIXED_PRECISION = "fp16"

# 3. ARCHITECTURE PARAMETERS FOR "WIDE-UNET" (Hitachi Replication)
# We maintain high capacity, but optimize memory management
BLOCK_OUT_CHANNELS = (256, 512, 512, 1024) # "Wide" criterion
LAYERS_PER_BLOCK = 2

# CRITERION 3: Matched GroupNorm
# Since we feed 16 channels, we use 16 groups for perfect separation
NUM_GROUPS = 16

print(f"✅ 'New Method' configuration with OOM protection loaded.")
print(f"📈 Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS} (Physical: {BATCH_SIZE})")
print(f"🏗️ Architecture: Wide-UNet {BLOCK_OUT_CHANNELS} | GroupNorm: {NUM_GROUPS}")

✅ 'New Method' Konfiguration mit OOM-Schutz geladen.
📈 Effektive Batch Size: 64 (Physikalisch: 16)
🏗️ Architektur: Wide-UNet (256, 512, 512, 1024) | GroupNorm: 16


In [ ]:
# Set the actual name of your "all-in-one" ZIP file here
ZIP_NAME = "bearing_all_in_one.zip"
ZIP_PATH = f"/content/drive/MyDrive/MasterProject/{ZIP_NAME}"

if os.path.exists(ZIP_PATH):
    print(f"🔍 Analyzing ZIP: {ZIP_NAME}...\n")

    # Display the first 15 files to inspect the structure
    print("--- Structure Check (First 15 lines) ---")
    !unzip -l {ZIP_PATH} | head -n 15

    print("\n--- Content Check (Summary) ---")
    # Count how many EnCodec files (latents) are included
    print("Number of EnCodec latents (.pt):")
    !unzip -l {ZIP_PATH} | grep "encodec" | grep ".pt" | wc -l

    # Count how many text embeddings are included
    print("Number of text embeddings (.pt):")
    !unzip -l {ZIP_PATH} | grep "text_embeddings" | grep ".pt" | wc -l

    # Check if 'bearing' folders exist
    print("\nDoes it contain 'bearing' data?")
    !unzip -l {ZIP_PATH} | grep -i "bearing" | head -n 5
else:
    print(f"❌ File not found: {ZIP_PATH}")
    print("Please check the filename in your Drive folder.")

🔍 Analysiere ZIP: bearing_all_in_one.zip...

--- Struktur-Check (Erste 15 Zeilen) ---
Archive:  /content/drive/MyDrive/MasterProject/bearing_all_in_one.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
   385985  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0057_vel_4_loc_C.pt
   385985  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0314_vel_4_loc_A.pt
   385985  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0853_vel_4_loc_B.pt
   385985  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0764_vel_4_loc_C.pt
   385992  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0000_vel_12_loc_A.pt
   385992  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0462_vel_12_loc_B.pt
   385992  2025-12-08 15:44   features/encodec/train/bearing/section_01_source_train_normal_0676_vel

In [ ]:
import os

# 1. Define paths
ZIP_PATH = '/content/drive/MyDrive/MasterProject/bearing_all_in_one.zip'
LOCAL_EXTRACT = '/content/dataset'

# 2. Copy and unzip
print("🚀 Copying All-in-One ZIP to local SSD...")
!cp {ZIP_PATH} /content/all_data.zip

print("📦 Unzipping 6599 pairs...")
!unzip -q /content/all_data.zip -d {LOCAL_EXTRACT}

# 3. FIX: Set final paths for training
# Based on your structure check, the data is now located here:
LATENT_ROOT = os.path.join(LOCAL_EXTRACT, 'features', 'encodec')
EMBEDDING_ROOT = os.path.join(LOCAL_EXTRACT, 'text_embeddings')

print("\n✅ DATA LOCALLY READY!")
print(f"Latent path: {LATENT_ROOT}")
print(f"Embedding path: {EMBEDDING_ROOT}")

# Short check if the 'train/bearing' directory exists
if os.path.exists(os.path.join(LATENT_ROOT, 'train', 'bearing')):
    print("👍 Directory structure verified.")
else:
    print("⚠️ Warning: Directory structure differs slightly. Check the file browser!")

🚀 Kopiere All-in-One ZIP auf lokale SSD...
📦 Entpacke 6599 Paare...

✅ DATEN LOKAL BEREIT!
Latent Pfad: /content/dataset/features/encodec
Embedding Pfad: /content/dataset/text_embeddings
👍 Ordnerstruktur verifiziert.


In [ ]:
import os
import glob
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class MIMIIDataset(Dataset):
    def __init__(self, latent_root, embedding_root, machine_filter='bearing'):
        self.data_pairs = []
        print(f"🔍 Searching for pairs in {latent_root}...")

        # 1. File search
        all_latents = glob.glob(os.path.join(latent_root, "**", "*.pt"), recursive=True)
        machine_latents = [f for f in all_latents if machine_filter.lower() in f.lower()]

        embedding_map = {os.path.splitext(os.path.basename(p))[0]: p
                         for p in glob.glob(os.path.join(embedding_root, "**", "*.pt"), recursive=True)}

        for lat_path in machine_latents:
            key = os.path.splitext(os.path.basename(lat_path))[0]
            if key in embedding_map:
                self.data_pairs.append((lat_path, embedding_map[key]))

        print(f"✅ {len(self.data_pairs)} pairs found for '{machine_filter}'.")

    def __len__(self):
        return len(self.data_pairs)

    def __getitem__(self, idx):
        l_path, e_path = self.data_pairs[idx]

        # Load to CPU
        latent = torch.load(l_path, map_location='cpu').squeeze(0)
        text_emb = torch.load(e_path, map_location='cpu').squeeze(0)

        # --- FIX 1: Reshape to 16 channels (Hitachi style) ---
        # 128 -> 16 channels with 8 frequency bins each
        latent_reshaped = latent.view(16, 8, 750)

        # --- FIX 2: The "holy padding" (750 -> 752) ---
        # Ensures perfect divisibility by 8 for the 3 downsampling stages
        latent_padded = F.pad(latent_reshaped, (0, 2), mode='constant', value=0)

        # --- FIX 3: Global normalization ---
        latent_norm = (latent_padded - GLOBAL_MEAN) / (GLOBAL_STD + 1e-6)

        return {"latent": latent_norm, "embedding": text_emb}

# --- CREATE DATALOADER ---
train_ds = MIMIIDataset(LATENT_ROOT, EMBEDDING_ROOT, machine_filter=MACHINE_TYPE)
train_size = int(0.9 * len(train_ds))
val_size = len(train_ds) - train_size
train_subset, val_subset = torch.utils.data.random_split(train_ds, [train_size, val_size])

# num_workers=4 and pin_memory=True for high-speed GPU data transfer
train_dl = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_dl = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"📊 Dataloader ready: {len(train_dl)} training batches, {len(val_dl)} validation batches.")

🔍 Suche Paare in /content/dataset/features/encodec...
✅ 6599 Paare für 'bearing' gefunden.
📊 Dataloader bereit: 372 Batches Training, 42 Batches Validierung.


In [ ]:
# ==============================================================================
# INITIALIZATION: NEUTRAL T5-EMBEDDING (The "joker" for 10% dropout) 🛠️
# ==============================================================================

import torch

# Set hardware (GPU if available, else CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using hardware: {device}")

print("🛠️ Generating valid null embedding for Classifier-Free Guidance...")

from transformers import T5Tokenizer, T5EncoderModel

# 1. Temporarily load encoder (must match architecture: flan-t5-base)
tmp_tok = T5Tokenizer.from_pretrained("google/flan-t5-base")
tmp_enc = T5EncoderModel.from_pretrained("google/flan-t5-base").to(device)

# 2. Encode empty string (IMPORTANT: max_length=64, just like your real data!)
null_input = tmp_tok("", return_tensors="pt", padding="max_length", max_length=64, truncation=True).to(device)

with torch.no_grad():
    # Extract the vector T5 outputs for "nothing"
    # This is not a zero tensor, but a mathematically correct T5 vector
    NULL_EMBEDDING = tmp_enc(null_input.input_ids).last_hidden_state.detach()

# 3. Free memory immediately
del tmp_tok, tmp_enc
torch.cuda.empty_cache()

print(f"✅ NULL_EMBEDDING successfully generated. Shape: {NULL_EMBEDDING.shape}")
print("   This embedding will now be used as a 'joker' in 10% of training cases.")

🚀 Nutze Hardware: cuda
🛠️ Generiere valides Null-Embedding für Classifier-Free Guidance...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

✅ NULL_EMBEDDING erfolgreich generiert. Shape: torch.Size([1, 64, 768])
   Dieses Embedding wird nun im Training zu 10% als 'Joker' genutzt.


In [ ]:
# ==============================================================================
# SCRIPT: STRATEGIC DATA SPLITTING (FOLDER-BASED)
# ==============================================================================
from torch.utils.data import Dataset, DataLoader, ConcatDataset # <--- Added ConcatDataset

# 1. DEFINE SOURCES (Strictly following your structure)
train_configs = [
    (os.path.join(LATENT_ROOT, 'train'), EMBEDDING_ROOT),
    (os.path.join(LATENT_ROOT, 'train_additional'), EMBEDDING_ROOT),
    (os.path.join(LATENT_ROOT, 'test/test_train'), EMBEDDING_ROOT)
]

val_configs = [
    (os.path.join(LATENT_ROOT, 'val'), EMBEDDING_ROOT),
    (os.path.join(LATENT_ROOT, 'val_additional'), EMBEDDING_ROOT),
    (os.path.join(LATENT_ROOT, 'test/test_val'), EMBEDDING_ROOT)
]

# 2. CREATE SUBSETS
def build_subset_list(config_list):
    subsets = []
    for l_dir, e_dir in config_list:
        if os.path.exists(l_dir):
            ds = MIMIIDataset(l_dir, e_dir, machine_filter=MACHINE_TYPE)
            if len(ds) > 0:
                subsets.append(ds)
    return subsets

train_subsets = build_subset_list(train_configs)
val_subsets = build_subset_list(val_configs)

# 3. CONCATENATE (Combine folder buckets)
full_train_ds = ConcatDataset(train_subsets)
full_val_ds = ConcatDataset(val_subsets)

# 4. DATALOADER (Randomization happens within the buckets)
train_dl = DataLoader(
    full_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,       # <--- Important: Random order during training
    num_workers=4,
    pin_memory=True
)

val_dl = DataLoader(
    full_val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,      # Validation does not need to be shuffled
    num_workers=4,
    pin_memory=True
)

print(f"\n✅ DATA STRUCTURE FINALIZED:")
print(f"   - Training: {len(full_train_ds)} samples (from train folders)")
print(f"   - Validation: {len(full_val_ds)} samples (from val folders)")
print(f"   - Ratio: {len(full_train_ds)/(len(full_train_ds)+len(full_val_ds)):.1%} / {len(full_val_ds)/(len(full_train_ds)+len(full_val_ds)):.1%}")

🔍 Suche Paare in /content/dataset/features/encodec/train...
✅ 2726 Paare für 'bearing' gefunden.
🔍 Suche Paare in /content/dataset/features/encodec/train_additional...
✅ 2730 Paare für 'bearing' gefunden.
🔍 Suche Paare in /content/dataset/features/encodec/test/test_train...
✅ 540 Paare für 'bearing' gefunden.
🔍 Suche Paare in /content/dataset/features/encodec/val...
✅ 273 Paare für 'bearing' gefunden.
🔍 Suche Paare in /content/dataset/features/encodec/val_additional...
✅ 270 Paare für 'bearing' gefunden.
🔍 Suche Paare in /content/dataset/features/encodec/test/test_val...
✅ 60 Paare für 'bearing' gefunden.

✅ DATENSTRUKTUR FINALISIERT:
   - Training: 5996 Samples (aus Train-Ordnern)
   - Validierung: 603 Samples (aus Val-Ordnern)
   - Verhältnis: 90.9% / 9.1%


In [ ]:
# ==============================================================================
# STEP 3: WIDE-UNET TRAINING WITH RESUME & 50-EPOCH CHECKPOINTS 🛡️⚙️
# ==============================================================================

import torch
import torch.nn.functional as F
from diffusers import UNet2DConditionModel, DDPMScheduler
from torch.optim import AdamW
from accelerate import Accelerator
from tqdm.auto import tqdm
import os
import numpy as np
import time

# --- 2.0 MODEL DEFINITION (If not already in memory) ---
if 'unet' not in locals():
    print("🏗️ UNet not found. Initializing Wide-UNet architecture...")
    unet = UNet2DConditionModel(
        sample_size=None,
        in_channels=16,
        out_channels=16,
        layers_per_block=LAYERS_PER_BLOCK,
        block_out_channels=BLOCK_OUT_CHANNELS, # (256, 512, 512, 1024)
        down_block_types=(
            "CrossAttnDownBlock2D", "CrossAttnDownBlock2D",
            "CrossAttnDownBlock2D", "CrossAttnDownBlock2D"
        ),
        up_block_types=(
            "CrossAttnUpBlock2D", "CrossAttnUpBlock2D",
            "CrossAttnUpBlock2D", "CrossAttnUpBlock2D"
        ),
        norm_num_groups=NUM_GROUPS, # Must be 16!
        cross_attention_dim=768,    # Matches Flan-T5
    ).to(device)

# --- 1. ACCELERATOR SETUP ---
accelerator = Accelerator(
    mixed_precision=MIXED_PRECISION,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS # Uses the "4"
)
device = accelerator.device

# --- 2. MODEL INITIALIZATION ---
# (If you created the model in the previous cell, it is prepared here)
optimizer = AdamW(unet.parameters(), lr=LEARNING_RATE)

# Noise Scheduler (Standard for Latent Diffusion)
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    variance_type="fixed_small",
    prediction_type="epsilon",
    clip_sample=False
)

# Prepare everything for accelerator
unet, optimizer, train_dl, val_dl = accelerator.prepare(unet, optimizer, train_dl, val_dl)

# --- 3. RESUME LOGIC & PATHS ---
RESUME_STATE_DIR = os.path.join(CHECKPOINT_DIR, "full_train_state")
os.makedirs(RESUME_STATE_DIR, exist_ok=True)

start_epoch = 0
loss_history = []
best_val_loss = float('inf')

# Check if we can resume training
if os.path.exists(os.path.join(RESUME_STATE_DIR, "scheduler.bin")):
    print(f"🔄 Checkpoint found! Loading state from: {RESUME_STATE_DIR}")
    accelerator.load_state(RESUME_STATE_DIR)

    # Read last epoch
    if os.path.exists(os.path.join(CHECKPOINT_DIR, "last_epoch.txt")):
        with open(os.path.join(CHECKPOINT_DIR, "last_epoch.txt"), "r") as f:
            start_epoch = int(f.read())

    # Load history
    if os.path.exists(os.path.join(CHECKPOINT_DIR, "loss_history.csv")):
        loss_history = np.loadtxt(os.path.join(CHECKPOINT_DIR, "loss_history.csv"), delimiter=",").tolist()
    print(f"▶️ Resuming training from epoch {start_epoch + 1}.")

# --- 4. THE TRAINING LOOP ---
print(f"🔥 Training running for {MACHINE_TYPE}...")

try:
    for epoch in range(start_epoch, NUM_EPOCHS):
        unet.train()
        train_loss = 0.0
        pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", disable=not accelerator.is_local_main_process)

        for batch in pbar:
            # Gradient accumulation mechanism
            with accelerator.accumulate(unet):
                latents = batch["latent"]
                embeddings = batch["embedding"]

                # --- 🎯 10% TEXT DROPOUT (CFG JOKER) ---
                if torch.rand(1) < 0.1:
                    # Uses the NULL_EMBEDDING from the previous cell
                    embeddings = NULL_EMBEDDING.expand(latents.shape[0], -1, -1)

                # --- 🌫️ DIFFUSION LOGIC ---
                noise = torch.randn_like(latents)
                timesteps = torch.randint(0, 1000, (latents.shape[0],), device=device).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Prediction & Loss
                noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=embeddings).sample
                loss = F.mse_loss(noise_pred, noise)

                # --- ⚖️ BACKWARD & CLIPPING ---
                accelerator.backward(loss)

                # Gradient Clipping: Prevents weights from "exploding"
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM) # MAX_GRAD_NORM = 1.0

                optimizer.step()
                optimizer.zero_grad()

                train_loss += loss.item()
                pbar.set_postfix({"MSE": f"{loss.item():.4f}"})

        avg_train_loss = train_loss / len(train_dl)
        loss_history.append(avg_train_loss)

        # --- 🧪 VALIDATION ---
        if (epoch + 1) % VALIDATION_INTERVAL == 0:
            unet.eval()
            val_loss = 0.0
            with torch.no_grad():
                for v_batch in val_dl:
                    v_lat, v_emb = v_batch["latent"], v_batch["embedding"]
                    v_noise = torch.randn_like(v_lat)
                    v_t = torch.randint(0, 1000, (v_lat.shape[0],), device=device).long()
                    v_noisy = noise_scheduler.add_noise(v_lat, v_noise, v_t)
                    v_pred = unet(v_noisy, v_t, encoder_hidden_states=v_emb).sample
                    val_loss += F.mse_loss(v_pred, v_noise).item()

            avg_val_loss = val_loss / len(val_dl)
            print(f"📊 Epoch {epoch+1}: Train-Loss {avg_train_loss:.5f} | Val-Loss {avg_val_loss:.5f}")

            # Save best model
            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_path = os.path.join(CHECKPOINT_DIR, "best_model.bin")
                accelerator.save(accelerator.unwrap_model(unet).state_dict(), best_path)
                print(f"🏆 Best model saved!")

        # --- 💾 MILESTONE SAVING (EVERY 50 EPOCHS) ---
        if (epoch + 1) % 50 == 0:
            print(f"Checkpoint milestone at epoch {epoch+1}...")

            # 1. Weights as a standalone file
            checkpoint_path = os.path.join(CHECKPOINT_DIR, f"model_epoch_{epoch+1}.bin")
            accelerator.save(accelerator.unwrap_model(unet).state_dict(), checkpoint_path)

            # 2. Full training state for resume (optimizer, scheduler, etc.)
            accelerator.save_state(RESUME_STATE_DIR)

            # 3. Metadata (history & last epoch)
            np.savetxt(os.path.join(CHECKPOINT_DIR, "loss_history.csv"), loss_history, delimiter=",")
            with open(os.path.join(CHECKPOINT_DIR, "last_epoch.txt"), "w") as f:
                f.write(str(epoch + 1))

            print(f"✅ Everything saved for epoch {epoch+1}.")

except Exception as e:
    print(f"❌ Error during training: {e}")

finally:
    # 10 min sync safety buffer
    print("\n🏁 Training paused. Waiting for Google Drive synchronization...")
    time.sleep(600)

🏗️ UNet nicht gefunden. Initialisiere Wide-UNet Architektur...
🔥 Training läuft für bearing...


Epoche 1/500:   0%|          | 0/375 [00:00<?, ?it/s]

Epoche 2/500:   0%|          | 0/375 [00:00<?, ?it/s]

Epoche 3/500:   0%|          | 0/375 [00:00<?, ?it/s]

Epoche 4/500:   0%|          | 0/375 [00:00<?, ?it/s]

...

Epoche 363/500:   0%|          | 0/375 [00:00<?, ?it/s]

Epoche 364/500:   0%|          | 0/375 [00:00<?, ?it/s]

Epoche 365/500:   0%|          | 0/375 [00:00<?, ?it/s]


🏁 Training pausiert. Warte auf Google Drive Synchronisation...


KeyboardInterrupt: 

In [ ]:
# ==============================================================================
# FINAL THESIS RUN: WIDE-UNET (RESUME 501 -> 700) 🚀
# ==============================================================================
import os
import torch
import torch.nn.functional as F
import numpy as np
import csv
import time
from tqdm.auto import tqdm
from diffusers import UNet2DConditionModel, DDPMScheduler
from torch.optim import AdamW
from accelerate import Accelerator
from google.colab import runtime

# --- 1. PATHS & CONFIGURATION ---
TARGET_EPOCH = 700
SAVE_INTERVAL = 50
HISTORY_CSV = os.path.join(CHECKPOINT_DIR, "full_detailed_history.csv")

# --- 2. INITIALIZATION ---
accelerator = Accelerator(mixed_precision=MIXED_PRECISION, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS)
device = accelerator.device

# ARCHITECTURE: Wide-UNet with 16 groups
unet = UNet2DConditionModel(
    sample_size=None,
    in_channels=16,
    out_channels=16,
    layers_per_block=LAYERS_PER_BLOCK,
    block_out_channels=BLOCK_OUT_CHANNELS,
    down_block_types=("CrossAttnDownBlock2D", "CrossAttnDownBlock2D", "CrossAttnDownBlock2D", "CrossAttnDownBlock2D"),
    up_block_types=("CrossAttnUpBlock2D", "CrossAttnUpBlock2D", "CrossAttnUpBlock2D", "CrossAttnUpBlock2D"),
    norm_num_groups=16, # <--- FIX: Exactly 16 groups for 16 channels
    cross_attention_dim=768,
)

optimizer = AdamW(unet.parameters(), lr=LEARNING_RATE)

# SCHEDULER: Scaled Linear & NO clipping
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_start=0.00085,
    beta_end=0.012,
    beta_schedule="scaled_linear",
    prediction_type="epsilon",
    clip_sample=False
)

# Preparation
unet, optimizer, train_dl, val_dl = accelerator.prepare(unet, optimizer, train_dl, val_dl)

# --- 3. RESUME LOGIC ---
if os.path.exists(os.path.join(RESUME_STATE_DIR, "optimizer.bin")):
    print(f"🔄 Checkpoint found! Loading state from epoch 500...")
    accelerator.load_state(RESUME_STATE_DIR)

    if os.path.exists(os.path.join(CHECKPOINT_DIR, "last_epoch.txt")):
        with open(os.path.join(CHECKPOINT_DIR, "last_epoch.txt"), "r") as f:
            start_epoch = int(f.read())
    else:
        start_epoch = 500
    print(f"▶️ Success! Continuing from epoch {start_epoch + 1}")
else:
    raise FileNotFoundError(f"❌ Error: No checkpoint found under {RESUME_STATE_DIR}!")

# --- 4. TRAINING & VALIDATION LOOP ---
try:
    for epoch in range(start_epoch, TARGET_EPOCH):
        unet.train()
        total_train_loss = 0.0
        pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{TARGET_EPOCH}")

        for batch in pbar:
            with accelerator.accumulate(unet):
                latents, embeddings = batch["latent"], batch["embedding"]

                # CFG Dropout (10%) with real NULL_EMBEDDING
                if torch.rand(1) < 0.1:
                    # Uses the NULL_EMBEDDING from the previous cell
                    embeddings = NULL_EMBEDDING.repeat(latents.shape[0], 1, 1).to(device)

                # --- 🌫️ DIFFUSION LOGIC ---
                noise = torch.randn_like(latents)
                timesteps = torch.randint(0, 1000, (latents.shape[0],), device=device).long()
                noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

                # Prediction & Loss
                noise_pred = unet(noisy_latents, timesteps, encoder_hidden_states=embeddings).sample
                loss = F.mse_loss(noise_pred, noise)

                # --- ⚖️ BACKWARD & CLIPPING ---
                accelerator.backward(loss)

                # Gradient Clipping: Prevents weights from "exploding"
                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(unet.parameters(), MAX_GRAD_NORM) # MAX_GRAD_NORM = 1.0

                optimizer.step()
                optimizer.zero_grad()
                total_train_loss += loss.item()
                pbar.set_postfix({"MSE": f"{loss.item():.4f}"})

        avg_train = total_train_loss / len(train_dl)

        # Validation
        unet.eval()
        total_val_loss = 0.0
        with torch.no_grad():
            for v_batch in val_dl:
                v_lat, v_emb = v_batch["latent"], v_batch["embedding"]
                v_noise = torch.randn_like(v_lat)
                v_t = torch.randint(0, 1000, (v_lat.shape[0],), device=device).long()
                v_noisy = noise_scheduler.add_noise(v_lat, v_noise, v_t)
                v_pred = unet(v_noisy, v_t, encoder_hidden_states=v_emb).sample
                total_val_loss += F.mse_loss(v_pred, v_noise).item()

        avg_val = total_val_loss / len(val_dl)

        # CSV Logging
        file_exists = os.path.isfile(HISTORY_CSV)
        with open(HISTORY_CSV, mode='a', newline='') as f:
            writer = csv.writer(f)
            if not file_exists: writer.writerow(["epoch", "train_mse", "val_mse"])
            writer.writerow([epoch + 1, avg_train, avg_val])

        print(f"📊 E{epoch+1}: Train-MSE {avg_train:.5f} | Val-MSE {avg_val:.5f}")

        # Saving
        if (epoch + 1) % SAVE_INTERVAL == 0:
            accelerator.save_state(RESUME_STATE_DIR)
            bin_path = os.path.join(CHECKPOINT_DIR, f"model_epoch_{epoch+1}.bin")
            accelerator.save(accelerator.unwrap_model(unet).state_dict(), bin_path)
            with open(os.path.join(CHECKPOINT_DIR, "last_epoch.txt"), "w") as f:
                f.write(str(epoch + 1))
            print(f"✅ Milestone {epoch+1} saved.")

except Exception as e:
    print(f"❌ Error in loop: {e}")

finally:
    print("🏁 Training finished. Waiting 60s for Drive sync...")
    time.sleep(60)
    print("🚀 Shutdown active.")
    runtime.unassign()

🔄 Checkpoint gefunden! Lade Zustand von Epoche 500...
▶️ Erfolg! Weiter ab Epoche 501


Epoche 501/700:   0%|          | 0/375 [00:00<?, ?it/s]

📊 E501: Train-MSE 0.02368 | Val-MSE 0.02647


Epoche 502/700:   0%|          | 0/375 [00:00<?, ?it/s]

📊 E502: Train-MSE 0.02373 | Val-MSE 0.02170


...

📊 E698: Train-MSE 0.02210 | Val-MSE 0.03207


Epoche 699/700:   0%|          | 0/375 [00:00<?, ?it/s]

📊 E699: Train-MSE 0.02061 | Val-MSE 0.03058


Epoche 700/700:   0%|          | 0/375 [00:00<?, ?it/s]

📊 E700: Train-MSE 0.02107 | Val-MSE 0.03034
✅ Meilenstein 700 gesichert.
🏁 Training beendet. Warte 60s auf Drive-Sync...
🚀 Shutdown aktiv.
